# 05 — Drug safety dashboard (offline)

Aggregates per-drug rating + ADR mention rate. The same logic powers the
`Drug safety dashboard` tab of the Streamlit app, but here we produce static
plots suitable for a portfolio README.

Drugs with N < 30 reviews are suppressed (small-sample noise).

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_parquet('../data/processed/clean.parquet')
adr = pd.read_parquet('../results/adr_mentions.parquet')
MIN_N = 30

In [ ]:
# per-drug aggregate
spans_per_review = adr.groupby('review_id').size().rename('n_spans').reset_index()
merged = df[['review_id', 'drug', 'rating']].merge(
    spans_per_review, on='review_id', how='left'
)
merged['has_adr'] = merged['n_spans'].fillna(0) > 0
agg = (merged.groupby('drug')
             .agg(n=('review_id', 'count'),
                  mean_rating=('rating', 'mean'),
                  adr_rate=('has_adr', 'mean'))
             .query('n >= @MIN_N')
             .sort_values('adr_rate', ascending=False))
agg.head(25)

In [ ]:
# scatter: rating vs ADR rate, size = N
fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(agg['mean_rating'], agg['adr_rate'], s=agg['n'].clip(upper=2000)/10, alpha=0.5)
ax.set_xlabel('Mean rating'); ax.set_ylabel('ADR mention rate')
ax.set_title('Drug-level: rating vs ADR mention rate (size = review count)')
for drug in agg.head(15).index:
    ax.annotate(drug, (agg.loc[drug, 'mean_rating'], agg.loc[drug, 'adr_rate']), fontsize=8)

In [ ]:
# per-condition negative-sentiment rate
cond = (df.dropna(subset=['condition'])
          .groupby('condition')
          .agg(n=('review_id','count'),
               neg_rate=('sentiment_3', lambda s: (s == 'negative').mean()))
          .query('n >= @MIN_N')
          .sort_values('neg_rate', ascending=False).head(25))
cond